In [3]:
from google import genai
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Load API key
load_dotenv("/Users/akshittyagi/Simply-Fit/.env")
gemini_key = os.getenv("GEMINI_API_KEY")

if gemini_key:
    print("Gemini API key loaded successfully")
else:
    print("Key not found — check your .env file")

# Configure Gemini client
client = genai.Client(api_key=gemini_key)

# Load dataset
df = pd.read_csv("../data/synthetic/users_weight_data.csv")
print(f"Dataset loaded: {df.shape}")

Gemini API key loaded successfully
Dataset loaded: (45000, 15)


In [4]:
# Build context string from user's real data
def build_user_context(user_id, df, target_weekly_change=-0.5):
    user_data      = df[df["user_id"] == user_id]
    weight_log     = user_data["weight"].values
    goal           = user_data["goal"].iloc[0]
    disease        = user_data["disease"].iloc[0]
    age            = user_data["age"].iloc[0]
    gender         = user_data["gender"].iloc[0]
    bmr            = user_data["bmr"].iloc[0]
    tdee           = user_data["tdee"].iloc[0]
    adherence      = user_data["adherence"].iloc[0]

    recent         = weight_log[-7:]
    weekly_change  = round(recent[-1] - recent[0], 2)
    current_weight = round(weight_log[-1], 1)
    start_weight   = round(weight_log[0], 1)
    total_change   = round(current_weight - start_weight, 2)
    kcal_balance   = round((weekly_change * 7700) / 7, 1)
    on_track       = abs(weekly_change - target_weekly_change) < 0.15

    context = f"""
USER PROFILE:
- Age: {age}, Gender: {gender}
- Current weight: {current_weight} kg
- Starting weight: {start_weight} kg
- Total change so far: {total_change:+} kg
- Goal: {goal} weight
- Medical condition: {disease}
- TDEE: {int(tdee)} kcal/day
- BMR: {int(bmr)} kcal/day

CURRENT WEEK METRICS:
- Weight change this week: {weekly_change:+} kg
- Target weekly change: {target_weekly_change:+} kg
- Estimated daily calorie balance: {kcal_balance:+.0f} kcal
- On track: {"Yes" if on_track else "No"}
- Adherence rate: {int(adherence * 100)}%

RECENT WEIGHT LOG (last 7 days):
{list(recent)}
"""
    return context

# Test
context = build_user_context(0, df)
print(context)


USER PROFILE:
- Age: 46, Gender: Male
- Current weight: 103.7 kg
- Starting weight: 105.3 kg
- Total change so far: -1.6 kg
- Goal: lose weight
- Medical condition: none
- TDEE: 3497 kcal/day
- BMR: 1840 kcal/day

CURRENT WEEK METRICS:
- Weight change this week: -0.4 kg
- Target weekly change: -0.5 kg
- Estimated daily calorie balance: -440 kcal
- On track: Yes
- Adherence rate: 43%

RECENT WEIGHT LOG (last 7 days):
[np.float64(104.1), np.float64(104.2), np.float64(104.1), np.float64(104.0), np.float64(103.7), np.float64(104.5), np.float64(103.7)]



In [7]:
# GenAI health coach using Gemini
def ask_coach(user_id, question, df, target_weekly_change=-0.5):
    
    context = build_user_context(user_id, df, target_weekly_change)
    
    # Extract key metrics for smart mock response
    user_data      = df[df["user_id"] == user_id]
    weekly_change  = round(user_data["weight"].values[-1] - 
                           user_data["weight"].values[-7], 2)
    goal           = user_data["goal"].iloc[0]
    on_track       = abs(weekly_change - target_weekly_change) < 0.15
    disease        = user_data["disease"].iloc[0]

    # Smart mock response based on real user data
    if on_track:
        response = (
            f"You are right on track. Your weight changed by "
            f"{weekly_change:+.2f} kg this week against a target of "
            f"{target_weekly_change:+.1f} kg. Keep doing what you are doing. "
            f"Consistency is your biggest asset right now."
        )
    else:
        gap = round(target_weekly_change - weekly_change, 2)
        response = (
            f"You are slightly off track this week — {weekly_change:+.2f} kg "
            f"vs target of {target_weekly_change:+.1f} kg. "
            f"Try closing the gap of {gap:+.2f} kg by reducing intake by "
            f"around {abs(int(gap * 7700 / 7 * 0.6))} kcal/day and increasing "
            f"activity by {abs(int(gap * 7700 / 7 * 0.4))} kcal/day."
        )

    if disease != "none":
        response += (
            f" Given your {disease} condition, please verify any "
            f"dietary changes with your doctor."
        )

    # TODO: Replace this block with real API call when credits available
    # response = client.models.generate_content(
    #     model="gemini-1.5-flash",
    #     contents=prompt
    # ).text

    return response

# Test
questions = [
    "Am I on track to reach my goal?",
    "Why is my weight not dropping faster?",
    "What should I focus on this week?"
]

for q in questions:
    print(f"Question: {q}")
    print(f"Coach: {ask_coach(0, q, df)}")
    print("-" * 60)

Question: Am I on track to reach my goal?
Coach: You are right on track. Your weight changed by -0.40 kg this week against a target of -0.5 kg. Keep doing what you are doing. Consistency is your biggest asset right now.
------------------------------------------------------------
Question: Why is my weight not dropping faster?
Coach: You are right on track. Your weight changed by -0.40 kg this week against a target of -0.5 kg. Keep doing what you are doing. Consistency is your biggest asset right now.
------------------------------------------------------------
Question: What should I focus on this week?
Coach: You are right on track. Your weight changed by -0.40 kg this week against a target of -0.5 kg. Keep doing what you are doing. Consistency is your biggest asset right now.
------------------------------------------------------------
